In [1]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin, urlparse

In [2]:
!pip install requests beautifulsoup4

In [3]:
def get_page(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; WikipediaScraper/1.0)"
    }

    response = requests.get(url, headers=headers, timeout=20)

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    return response.text, soup

In [4]:
url = "https://en.wikipedia.org/wiki/Python_(programming_language)"

html, soup = get_page(url)

print("HTML successfully retrieved!")
print("HTML length:", len(html))

HTML successfully retrieved!
HTML length: 1021460


In [5]:
def extract_title(soup):
    title = soup.find("h1", id="firstHeading")

    if title:
        return title.get_text(strip=True)

    return "Title not found"

In [6]:
title = extract_title(soup)

print("Article Title:", title)

Article Title: Python (programming language)


In [7]:
{
    "Introduction": [
        "Python is a high-level programming language..."
    ],

    "History": [
        "Python was conceived in the late 1980s..."
    ],

    "Syntax and semantics": [
        "Python uses indentation..."
    ]
}

{'Introduction': ['Python is a high-level programming language...'],
 'History': ['Python was conceived in the late 1980s...'],
 'Syntax and semantics': ['Python uses indentation...']}

In [8]:
def extract_article_text(soup):

    article = soup.find("div", id="mw-content-text")

    content = article.find("div", class_="mw-parser-output")

    sections = {}

    current_heading = "Introduction"

    sections[current_heading] = []

    for element in content.find_all(
        ["h2", "h3", "h4", "p"],
        recursive=True
    ):

        if element.name in ["h2", "h3", "h4"]:

            heading = element.get_text(" ", strip=True)

            heading = re.sub(r"\[edit\]", "", heading).strip()

            if heading:
                current_heading = heading

                if current_heading not in sections:
                    sections[current_heading] = []

        elif element.name == "p":

            paragraph = element.get_text(" ", strip=True)

            if paragraph:
                sections[current_heading].append(paragraph)

    return sections

In [9]:
article_text = extract_article_text(soup)

for heading, paragraphs in article_text.items():

    print("\n", heading)

    for paragraph in paragraphs[:2]:
        print("-", paragraph[:200])


 Introduction
- Python is a high-level , general-purpose programming language that emphasizes code readability , simplicity, and ease-of-writing with the use of significant indentation , [ 38 ] an extensive ("batteri
- Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible wit

 History
- Python was conceived in the late 1980s [ 11 ] by Guido van Rossum at Centrum Wiskunde & Informatica (CWI) in the Netherlands . [ 43 ] It was designed as a successor to the ABC programming language, wh
- The name Python derives from the British comedy series Monty Python's Flying Circus . [ 48 ] (See § Naming .)

 Design philosophy and features
- Python is a multi-paradigm programming language . Object-oriented programming and structured programming are fully supported, and many of their features support functional programming and aspect-orien
- Python uses dyna

In [10]:
def extract_internal_links(soup, base_url):

    links = set()

    for link in soup.find_all("a", href=True):

        href = link["href"]

        if href.startswith("/wiki/"):

            full_url = urljoin(base_url, href)

            parsed_url = urlparse(full_url)

            if parsed_url.netloc.endswith("wikipedia.org"):

                if ":" not in parsed_url.path.split("/wiki/")[-1]:

                    clean_url = full_url.split("#")[0]

                    links.add(clean_url)

    return sorted(links)

In [11]:
internal_links = extract_internal_links(soup, url)

print("Total internal Wikipedia links:", len(internal_links))

print("\nFirst 10 links:")

for link in internal_links[:10]:
    print(link)

Total internal Wikipedia links: 2

First 10 links:
https://en.wikipedia.org/wiki/Main_Page
https://en.wikipedia.org/wiki/Python_(programming_language)


In [12]:
def scrape_wikipedia(url):

    # Step 1: Retrieve and parse HTML
    html, soup = get_page(url)

    # Step 2: Extract article title
    title = extract_title(soup)

    # Step 3: Extract article text
    article_text = extract_article_text(soup)

    # Step 4: Extract internal Wikipedia links
    internal_links = extract_internal_links(soup, url)

    # Step 5: Combine everything into one dictionary
    result = {
        "html": html,
        "title": title,
        "article_text": article_text,
        "internal_links": internal_links
    }

    return result

In [13]:
scrape_wikipedia(url)

{'html': '<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard skin-underline-clientpref-browser" lang="en" dir="ltr">\n<head>\n<meta charset="UTF-8">\n<title>Python (programming language) - Wikipedia</title>\n<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pi

In [14]:
# Select a Wikipedia page
test_url = "https://en.wikipedia.org/wiki/Python_(programming_language)"

# Run the consolidated scraping function
data = scrape_wikipedia(test_url)

# Display the results
print("ARTICLE TITLE:")
print(data["title"])

print("\nNUMBER OF HEADINGS:")
print(len(data["article_text"]))

print("\nNUMBER OF INTERNAL LINKS:")
print(len(data["internal_links"]))

print("\nFIRST 5 INTERNAL LINKS:")

for link in data["internal_links"][:5]:
    print(link)

ARTICLE TITLE:
Python (programming language)

NUMBER OF HEADINGS:
29

NUMBER OF INTERNAL LINKS:
2

FIRST 5 INTERNAL LINKS:
https://en.wikipedia.org/wiki/Main_Page
https://en.wikipedia.org/wiki/Python_(programming_language)


In [15]:
print("HTML retrieved:", len(data["html"]) > 0)

print("\nHTML preview:")
print(data["html"][:500])

HTML retrieved: True

HTML preview:
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp
